# Примеры применения SiLK Suite через PySiLK

По мотивам главы 9 «The SiLK Suite» из книги:  
> Michael Collins. *Network Security Through Data Analysis: From Data to Action*. 2nd ed. O'Reilly Media, 2017. ISBN 978-1-491-96284-8.

**Источники данных для примеров:**
- CERT SiLK Repository (учебные данные Cyber Defense Exercise): https://tools.netsa.cert.org/silk/referencedata.html
- Документация PySiLK: https://tools.netsa.cert.org/silk/pySiLK.html
- Исходный код SiLK (включает PySiLK): https://tools.netsa.cert.org/silk/download.html

## Установка и настройка окружения

PySiLK поставляется вместе с пакетом SiLK и доступен после его сборки.  
Переменные окружения, которые должны быть установлены:
- `SILK_DATA_ROOTDIR` — путь к хранилищу flow-данных  
- `SILK_CONFIG_FILE` — путь к конфигурационному файлу `silk.conf`

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    libpcap-dev \
    libglib2.0-dev \
    libgnutls28-dev \
    zlib1g-dev \
    python3-dev \
    pkg-config \
    wget

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
(Reading database ... 117540 files and directories currently installed.)
Removing r-base-dev (4.5.2-1.2204.0) ...
(Reading database ... 117538 files and directories currently installed.)
Preparing to unpack .../libglib2.0-dev_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-dev:amd64 (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../libglib2.0-dev-bin_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-dev-bin (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../libglib2.0-bin_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-bin (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../libglib2.0-0_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-0:amd64 (2.72.4-0ubuntu2.9) over (2.72.4-0ub

In [8]:
!wget https://tools.netsa.cert.org/releases/libfixbuf-2.5.4.tar.gz -O /tmp/libfixbuf.tar.gz

--2026-03-17 21:43:47--  https://tools.netsa.cert.org/releases/libfixbuf-2.5.4.tar.gz
Resolving tools.netsa.cert.org (tools.netsa.cert.org)... 147.72.252.217
Connecting to tools.netsa.cert.org (tools.netsa.cert.org)|147.72.252.217|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1036011 (1012K) [application/x-gzip]
Saving to: ‘/tmp/libfixbuf.tar.gz’

/tmp/libfixbuf.tar. 100%[===================>]   1012K   281KB/s    in 3.7s    

2026-03-17 21:43:51 (276 KB/s) - ‘/tmp/libfixbuf.tar.gz’ saved [1036011/1036011]



In [9]:
!tar -xzf /tmp/libfixbuf.tar.gz -C /tmp

!cd /tmp/libfixbuf-2.5.4 && \
    ./configure --prefix=/usr/local \
                --quiet \
                2>&1 | tail -3 && \
    make -j$(nproc) -s && \
    make install -s

!ldconfig

    * Linker flags (LDFLAGS):       
    * Libraries (LIBS):             -lpthread 

Making all in src
Making all in infomodel
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I../include -Wall -Wextra -Wshadow -Wpointer-arith -Wformat=2 -Wunused -Wduplicated-cond -Wwrite-strings -Wmissing-prototypes -Wstrict-prototypes -DNDEBUG -DG_DISABLE_ASSERT -pthread -I/usr/include/glib-2.0 -I/usr/lib/x86_64-linux-gnu/glib-2.0/include -g -O2 -MT fbinfomodel.lo -MD -MP -MF .deps/fbinfomodel.Tpo -c fbinfomodel.c  -fPIC -DPIC -o .libs/fbinfomodel.o
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I../include -Wall -Wextra -Wshadow -Wpointer-arith -Wformat=2 -Wunused -Wduplicated-cond -Wwrite-strings -Wmissing-prototypes -Wstrict-prototypes -DNDEBUG -DG_DISABLE_ASSERT -pthread -I/usr/include/glib-2.0 -I/usr/lib/x86_64-linux-gnu/glib-2.0/include -g -O2 -MT fbuf.lo -MD -MP -MF .deps/fbuf.Tpo -c fbuf.c  -fPIC -DPIC -o .libs/fbuf.o
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I../include -Wall -Wextra -Wshadow -Wpo

In [10]:
SILK_VER = '3.24.0'
SILK_URL = f'https://tools.netsa.cert.org/releases/silk-{SILK_VER}.tar.gz'

SILK_DATA_DIR = '/opt/silk/data'
SILK_PREFIX   = '/usr/local'

!wget {SILK_URL} -O /tmp/silk.tar.gz
!tar -xzf /tmp/silk.tar.gz -C /tmp

!cd /tmp/silk-{SILK_VER} && \
    ./configure \
        --prefix={SILK_PREFIX} \
        --enable-python \
        --with-python=/usr/bin/python3 \
        --with-fixbuf \
        --with-libpcap \
        --with-zlib \
        --enable-ipv6 \
        --quiet \
        CFLAGS='-O2' \
        2>&1 | tail -5 && \
    make -j$(nproc) -s && \
    make install -s

!ldconfig


    * Compiler flags (CFLAGS):      -I$(srcdir) -I$(top_builddir)/src/include -I$(top_srcdir)/src/include -DNDEBUG -D_ALL_SOURCE=1 -D_GNU_SOURCE=1 -Wall -Wextra -Wmissing-prototypes -Wformat=2 -Wdeclaration-after-statement -Wpointer-arith -fno-strict-aliasing -O2
    * Linker flags (LDFLAGS):       
    * Libraries (LIBS):             -lz -lm

configure: WARNING: unrecognized options: --enable-python, --with-fixbuf, --with-libpcap
Making all in tests
Making all in src
Making all in libsilk
libtool: compile:  gcc -I. -I../../src/include -I../../src/include -DNDEBUG -D_ALL_SOURCE=1 -D_GNU_SOURCE=1 -DSILK_DATA_ROOTDIR=\"/data\" -DSILK_PYTHON_SITE_PKG=\"/usr/local/lib/python3.12/dist-packages\" -DSILK_PYTHON_VERSION=\"3.12.12\" -Wall -Wextra -Wmissing-prototypes -Wformat=2 -Wdeclaration-after-statement -Wpointer-arith -fno-strict-aliasing -O2 -MT addrtype.lo -MD -MP -MF .deps/addrtype.Tpo -c addrtype.c  -fPIC -DPIC -o .libs/addrtype.o
libtool: compile:  gcc -I. -I../../src/include -I../../

In [11]:
import os
os.makedirs(SILK_DATA_DIR, exist_ok=True)
print(f'✓ SiLK {SILK_VER} установлен (с PySiLK и поддержкой IPv6)')

# Проверить наличие CLI-утилит
!which rwfilter rwcut rwcount rwset rwuniq 2>&1

✓ SiLK 3.24.0 установлен (с PySiLK и поддержкой IPv6)
/usr/local/bin/rwfilter
/usr/local/bin/rwcut
/usr/local/bin/rwcount
/usr/local/bin/rwset
/usr/local/bin/rwuniq


In [12]:
import os, sys
import site

SILK_PREFIX   = '/usr/local'
SILK_DATA_DIR = '/opt/silk/data'

# Переменные окружения, нужные SiLK и PySiLK
os.environ['SILK_DATA_ROOTDIR'] = SILK_DATA_DIR
os.environ['SILK_CONFIG_FILE']  = f'{SILK_DATA_DIR}/silk.conf'

# Добавить путь к PySiLK в sys.path
# PySiLK устанавливается в site-packages текущего Python
pyver = f'{sys.version_info.major}.{sys.version_info.minor}'
silk_site = f'{SILK_PREFIX}/lib/python{pyver}/site-packages'
if silk_site not in sys.path:
    sys.path.insert(0, silk_site)
os.environ['PYTHONPATH'] = silk_site + ':' + os.environ.get('PYTHONPATH', '')

# Минимальный silk.conf для одного класса/типа (достаточно для примеров)
SILK_CONF = """# Минимальная конфигурация SiLK для Google Colab
version 2
sensor 0 S0 "Sensor 0"
class all
    type  0 in      in
    type  1 out     out
    type  2 inweb   inweb
    type  3 outweb  outweb
    sensors S0
    default-types in out inweb outweb
end class
default-class all
"""

with open(os.environ['SILK_CONFIG_FILE'], 'w') as f:
    f.write(SILK_CONF)
print(f"✓ silk.conf записан: {os.environ['SILK_CONFIG_FILE']}")

# Создать дерево директорий репозитория
!mkdir -p {SILK_DATA_DIR}/in/2009/02/12
!mkdir -p {SILK_DATA_DIR}/out/2009/02/12
!mkdir -p {SILK_DATA_DIR}/inweb/2009/02/12
!mkdir -p {SILK_DATA_DIR}/outweb/2009/02/12
print(f'✓ Дерево данных создано: {SILK_DATA_DIR}')

✓ silk.conf записан: /opt/silk/data/silk.conf
✓ Дерево данных создано: /opt/silk/data


In [18]:
import silk
from silk import silkfile_open, READ, WRITE, IPSet, IPWildcard, IPAddr
import silk.site

print(f'✓ PySiLK импортирован успешно')
print(f'  Версия SiLK:    {silk.silk_version()}')
#print(f'  Поддержка IPv6: {silk.silk_is_ipv6()}')

# Инициализировать site (читает SILK_CONFIG_FILE)
silk.site.init_site()
print(f'  Классы сенсоров: {list(silk.site.classes())}')
#print(f'  Типы трафика:   {list(silk.site.flowtypes())}')

# Быстрый smoke-test: создать IPSet, записать/прочитать SiLK-файл
import os, tempfile

test_set = IPSet(IPWildcard('192.168.1.0/24'))
test_set.add(IPAddr('10.0.0.1'))
assert IPAddr('192.168.1.42') in test_set
print('✓ IPSet работает корректно')

# Записать и прочитать тестовый .rw-файл
test_file = '/tmp/test_silk.rw'
with silkfile_open(test_file, WRITE) as f:
    rec = silk.RWRec()
    rec.sip     = IPAddr('10.0.0.1')
    rec.dip     = IPAddr('8.8.8.8')
    rec.sport   = 54321
    rec.dport   = 443
    rec.protocol = 6
    rec.bytes   = 1024
    rec.packets = 10
    f.write(rec)

with silkfile_open(test_file, READ) as f:
    for r in f:
        print(f'✓ Тестовая запись: {r.sip} → {r.dip}:{r.dport}  {r.bytes} bytes')

os.remove(test_file)
print('\n✅ Всё готово — можно запускать примеры из следующих ячеек!')

✓ PySiLK импортирован успешно
  Версия SiLK:    3.24.0
  Классы сенсоров: ['all']
✓ IPSet работает корректно
✓ Тестовая запись: 10.0.0.1 → 8.8.8.8:443  1024 bytes

✅ Всё готово — можно запускать примеры из следующих ячеек!


In [19]:
YAF_VER = '2.18.2'
YAF_URL = f'https://tools.netsa.cert.org/releases/yaf-{YAF_VER}.tar.gz'

!wget -q {YAF_URL} -O /tmp/yaf.tar.gz
!tar -xzf /tmp/yaf.tar.gz -C /tmp

!cd /tmp/yaf-{YAF_VER} && \
    ./configure \
        --prefix=/usr/local \
        --with-libpcap \
        --quiet \
        2>&1 | tail -3 && \
    make -j$(nproc) -s && \
    make install -s

!which yaf && yaf --version 2>&1 | head -2

    * Linker flags (LDFLAGS):       -lpcre
    * Libraries (LIBS):             -lpcap -lm -lz

Making all in libltdl
  CC       libltdlc_la-lt__alloc.lo
  CC       libltdlc_la-lt_dlloader.lo
  CC       libltdlc_la-lt_error.lo
  CC       libltdlc_la-ltdl.lo
  CC       libltdlc_la-slist.lo
  CC       lt__strl.lo
  CC       loaders/dlopen.lo
  CC       loaders/libltdlc_la-preopen.lo
  CCLD     dlopen.la
  CCLD     libltdlc.la
Making all in infomodel
Making all in lua
Making all in src
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -g -O2 -MT lapi.lo -MD -MP -MF .deps/lapi.Tpo -c lapi.c  -fPIC -DPIC -o .libs/lapi.o
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -g -O2 -MT lauxlib.lo -MD -MP -MF .deps/lauxlib.Tpo -c lauxlib.c  -fPIC -DPIC -o .libs/lauxlib.o
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -g -O2 -MT lauxlib.lo -MD -MP -MF .deps/lauxlib.Tpo -c lauxlib.c -o lauxlib.o >/dev/null 2>&1
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -g -O2 -MT lapi.lo -MD -MP -MF .deps/lapi.Tpo

In [27]:
import os
import sys
from datetime import datetime, timedelta
from collections import defaultdict

# Базовые зависимости для анализа и визуализации
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Основной модуль PySiLK
# Требует установленного SiLK с поддержкой Python-биндингов
import silk
from silk import (
    silkfile_open, READ, WRITE, APPEND,
    IPSet, IPWildcard, IPAddr, silk_version
)
import silk.site

print(f"PySiLK version: {silk_version()}")
#print(f"IPv6 support:   {silk_is_ipv6()}")

# Укажите путь к вашему хранилищу данных
# Для учебных данных CERT CDX:
# os.environ['SILK_DATA_ROOTDIR'] = '/path/to/silk/data'
# os.environ['SILK_CONFIG_FILE']  = '/path/to/silk.conf'

# Инициализация site (читает SILK_CONFIG_FILE)
#silk.site.init_site()

PySiLK version: 3.24.0


---
## Пример 1. Просмотр flow-записей (`rwcut`)

**Задача:** прочитать SiLK-файл и вывести записи в читаемом виде — аналог `rwcut --fields=...`

Оригинальная команда:
```bash
rwcut --fields=sIP,dIP,sPort,dPort,proto,bytes,packets,duration data.rw
rwcut --fields=sIP,dIP,sPort,dPort,proto --num-recs=10 data.rw
rwcut --fields=sIP,dIP,sPort,dPort,proto --no-titles --delimited=, data.rw
```

**Источник данных:** файл `data.rw` из репозитория CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [31]:
!rwcut --num-recs=10 /content/drive/MyDrive/DataAnalysis/in-S0_20041004.20

                                    sIP|                                    dIP|sPort|dPort|pro|     packets|          bytes|   flags|                     sTime|    duration|                     eTime|sen|
                         203.13.173.243|                            128.3.45.10|   53| 2124| 17|           1|             64|        |2004/10/04T20:05:02.331000|    0.000000|2004/10/04T20:05:02.331000| S0|
                           33.115.84.19|                           128.3.46.146| 4087| 5554|  6|           1|             48| S      |2004/10/04T20:11:48.675000|    0.002000|2004/10/04T20:11:48.677000| S0|
                           33.115.84.19|                           128.3.46.146| 4434| 9898|  6|           1|             48| S      |2004/10/04T20:11:49.497000|    0.001000|2004/10/04T20:11:49.498000| S0|
                         148.184.175.97|                           128.3.44.112|  389| 1549| 17|           1|            201|        |2004/10/04T20:07:18.685000|    0.000000|20

In [29]:
# Путь к учебному файлу данных (измените на свой)
DATA_FILE = '/content/drive/MyDrive/DataAnalysis/in-S0_20041004.20'

# --- Шаг 1: открыть файл и считать записи ---
records = []
with silkfile_open(DATA_FILE, READ) as f:
    for rec in f:
        records.append({
            'sIP':      str(rec.sip),
            'dIP':      str(rec.dip),
            'sPort':    rec.sport,
            'dPort':    rec.dport,
            'proto':    rec.protocol,
            'bytes':    rec.bytes,
            'packets':  rec.packets,
            'duration': rec.duration,   # в миллисекундах
            'sTime':    rec.stime,      # объект datetime
        })

df = pd.DataFrame(records)
print(f"Всего записей: {len(df)}")

# --- Шаг 2: вывести первые 10 записей (аналог --num-recs=10) ---
print("\n--- Первые 10 записей ---")
print(df[['sIP', 'dIP', 'sPort', 'dPort', 'proto', 'bytes', 'packets', 'duration']].head(10).to_string(index=False))

# --- Шаг 3: вывести как CSV (аналог --no-titles --delimited=,) ---
print("\n--- CSV-формат (без заголовков) ---")
print(df[['sIP', 'dIP', 'sPort', 'dPort', 'proto']].head(5).to_csv(index=False, header=False))

Всего записей: 1470

--- Первые 10 записей ---
            sIP          dIP  sPort  dPort  proto  bytes  packets               duration
 203.13.173.243  128.3.45.10     53   2124     17     64        1        0 days 00:00:00
   33.115.84.19 128.3.46.146   4087   5554      6     48        1 0 days 00:00:00.002000
   33.115.84.19 128.3.46.146   4434   9898      6     48        1 0 days 00:00:00.001000
 148.184.175.97 128.3.44.112    389   1549     17    201        1        0 days 00:00:00
 148.184.175.97 128.3.44.112      0      0      1    120        2 0 days 00:00:00.182000
207.235.115.253 128.3.44.112   5190   4973      6     80        2 0 days 00:05:00.267000
207.235.255.108 128.3.44.112   5002   4974      6    767       10 0 days 00:05:03.024000
148.184.191.214 128.3.44.112      0    771      1    112        2 0 days 00:01:55.138000
  148.184.171.6 128.3.44.112    389   1560     17    197        1        0 days 00:00:00
 148.184.175.97 128.3.44.112    389   1561     17    201       

---
## Пример 2. Фильтрация трафика по порту и протоколу (`rwfilter`)

**Задача:** выделить все TCP-потоки на порт 80 (HTTP) — аналог `rwfilter --proto=6 --dport=80`

Оригинальная команда:
```bash
rwfilter --start-date=2004/10/04 \
         --proto=6 --dport=80 \
         --pass=http_traffic.rw --fail=/dev/null
```

**Источник данных:** учебный набор CERT CDX, дата 2009-02-12  
https://tools.netsa.cert.org/silk/repository.html

In [53]:
!rwfilter  --dport=53 \
          /content/drive/MyDrive/DataAnalysis/in-S0_20041004.20 \
         --pass=http_traffic.rw --fail=/dev/null

In [52]:
!rm -f http_traffic.rw

In [54]:
!rwcut --num-recs=10 http_traffic.rw

                                    sIP|                                    dIP|sPort|dPort|pro|     packets|          bytes|   flags|                     sTime|    duration|                     eTime|sen|
                          35.207.78.143|                           128.3.23.158|41247|   53| 17|           1|            210|        |2004/10/04T20:11:57.951000|    0.000000|2004/10/04T20:11:57.951000| S0|
                         201.68.209.242|                        131.243.142.235|   53|   53| 17|           1|             52|        |2004/10/04T20:16:22.117000|    0.000000|2004/10/04T20:16:22.117000| S0|


In [32]:
# --- Шаг 1: задать параметры запроса ---
TARGET_DATE_STR = '2009/02/12'
TARGET_DATE     = datetime.strptime(TARGET_DATE_STR, '%Y/%m/%d')
PROTO_TCP       = 6
PORT_HTTP       = 80
OUTPUT_FILE     = 'http_traffic.rw'

# --- Шаг 2: получить список файлов из репозитория за указанную дату ---
# silk.site.repository_iter возвращает пути к .rw-файлам по параметрам
repo_files = list(silk.site.repository_iter(
    start_date=TARGET_DATE,
    end_date=TARGET_DATE,
    flowtypes=None   # все типы трафика (in, out, inweb, outweb и т.д.)
))
print(f"Файлов в репозитории за {TARGET_DATE_STR}: {len(repo_files)}")

# --- Шаг 3: фильтрация — аналог --proto=6 --dport=80 --pass=... ---
http_records = []
total_read   = 0

with silkfile_open(OUTPUT_FILE, WRITE) as out_f:
    for path in repo_files:
        with silkfile_open(path, READ) as in_f:
            for rec in in_f:
                total_read += 1
                # Условие фильтрации: TCP + порт назначения 80
                if rec.protocol == PROTO_TCP and rec.dport == PORT_HTTP:
                    out_f.write(rec)
                    http_records.append({
                        'sIP':   str(rec.sip),
                        'dIP':   str(rec.dip),
                        'sPort': rec.sport,
                        'dPort': rec.dport,
                        'bytes': rec.bytes,
                        'sTime': rec.stime,
                    })

df_http = pd.DataFrame(http_records)
print(f"Прочитано всего:     {total_read} записей")
print(f"Отфильтровано HTTP:  {len(df_http)} записей")
print(f"Записано в файл:     {OUTPUT_FILE}")
print()

# --- Шаг 4: проверить результат — первые строки ---
print(df_http.head(10).to_string(index=False))

TypeError: repository_iter() got an unexpected keyword argument 'start_date'

In [ ]:
# Вспомогательная функция для повторного использования фильтрации
# (аналог конвейера rwfilter ... | rwcut ...)

def filter_flows(repo_files, proto=None, dport=None, sip_range=None, dip_range=None):
    """Генератор: возвращает записи, удовлетворяющие условиям фильтра."""
    sip_wc = IPWildcard(sip_range) if sip_range else None
    dip_wc = IPWildcard(dip_range) if dip_range else None

    for path in repo_files:
        with silkfile_open(path, READ) as f:
            for rec in f:
                if proto      is not None and rec.protocol != proto:  continue
                if dport      is not None and rec.dport    != dport:  continue
                if sip_wc     and rec.sip not in sip_wc:              continue
                if dip_wc     and rec.dip not in dip_wc:              continue
                yield rec

# Пример использования — HTTP-трафик через генератор:
count = sum(1 for _ in filter_flows(repo_files, proto=6, dport=80))
print(f"HTTP-потоков (через генератор): {count}")

---
## Пример 3. Подсчёт объёма трафика по времени (`rwcount`)

**Задача:** построить временной ряд байт/потоков для обнаружения аномалий —  
аналог `rwcount --bin-size=3600`.

Оригинальная команда:
```bash
rwfilter --start-date=2009/02/12 --daddress=10.0.0.5 \
         --pass=stdout --fail=/dev/null | \
rwcount --bin-size=3600
```

**Источник данных:** учебный набор CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
from datetime import timezone

# --- Шаг 1: задать параметры ---
BIN_SIZE_SEC  = 3600        # 1 час
TARGET_SERVER = '10.0.0.5'  # IP сервера, трафик к которому анализируем

server_ip = IPAddr(TARGET_SERVER)

# --- Шаг 2: агрегировать записи по временным бинам ---
bins_flows   = defaultdict(int)
bins_bytes   = defaultdict(int)
bins_packets = defaultdict(int)

for path in repo_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            # Фильтр: только трафик к целевому серверу
            if rec.dip != server_ip:
                continue
            # Вычислить ключ бина: время начала потока, округлённое до BIN_SIZE_SEC
            ts  = rec.stime.timestamp()
            bin_key = datetime.fromtimestamp(
                (ts // BIN_SIZE_SEC) * BIN_SIZE_SEC, tz=timezone.utc
            )
            bins_flows[bin_key]   += 1
            bins_bytes[bin_key]   += rec.bytes
            bins_packets[bin_key] += rec.packets

# --- Шаг 3: собрать в DataFrame ---
df_count = pd.DataFrame({
    'datetime': sorted(bins_flows.keys()),
    'Flows':    [bins_flows[k]   for k in sorted(bins_flows)],
    'Bytes':    [bins_bytes[k]   for k in sorted(bins_bytes)],
    'Packets':  [bins_packets[k] for k in sorted(bins_packets)],
})

print(df_count.to_string(index=False))

In [ ]:
# --- Шаг 4: построить временной ряд (как в главе 11 книги) ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].bar(df_count['datetime'], df_count['Bytes'] / 1e6, width=0.03, color='steelblue')
axes[0].set_ylabel('Мегабайты')
axes[0].set_title(f'Трафик к {TARGET_SERVER} по часам')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_count['datetime'], df_count['Flows'], width=0.03, color='darkorange')
axes[1].set_ylabel('Потоков')
axes[1].set_xlabel('Время')
axes[1].grid(axis='y', alpha=0.3)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.savefig('rwcount_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Шаг 5: базовая статистика для построения аномального детектора ---
print("\nСтатистика по байтам (за все бины):")
print(df_count['Bytes'].describe())

threshold_95 = df_count['Bytes'].quantile(0.95)
print(f"\n95-й перцентиль (порог тревоги): {threshold_95:,.0f} байт")
anomalous = df_count[df_count['Bytes'] > threshold_95]
print(f"Бинов, превышающих порог:         {len(anomalous)}")

---
## Пример 4. IP-множества: создание и использование в фильтрах (`rwset`)

**Задача:** сформировать список подозрительных IP-адресов (сканеров «тёмных адресов») и  
использовать его как фильтр — аналог `rwset`, `rwsetcat`, `rwsettool`.

Оригинальные команды:
```bash
rwfilter ... --daddress=10.99.0.0/16 --pass=stdout | rwset --sip-file=scanners.set
rwsetcat scanners.set
rwsettool --union known_bad.set scanners.set > combined.set
rwfilter ... --sipset=scanners.set --pass=scanner_activity.rw
```

**Источник данных:** учебный набор CERT CDX (тёмные адреса — неиспользуемое адресное пространство)  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# --- Шаг 1: создать IP-множество сканеров по трафику к «тёмным адресам» ---
DARK_NET = '10.99.0.0/16'  # неиспользуемое адресное пространство
dark_wc  = IPWildcard(DARK_NET)

scanners_set = IPSet()

for path in repo_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            if rec.dip in dark_wc:
                scanners_set.add(rec.sip)  # источник -> подозрительный

print(f"Обнаружено уникальных сканеров: {len(scanners_set)}")

# --- Шаг 2: просмотреть содержимое множества (аналог rwsetcat) ---
print("\nПервые 20 адресов из множества:")
for i, ip in enumerate(scanners_set):
    print(f"  {ip}")
    if i >= 19:
        break

# --- Шаг 3: сохранить множество в файл ---
scanners_set.save('scanners.set')
print("\nМножество сохранено: scanners.set")

In [ ]:
# --- Шаг 4: загрузить существующее множество и объединить (аналог rwsettool --union) ---
# Предположим, у нас есть файл known_bad.set с известными вредоносными адресами
try:
    known_bad_set = IPSet.load('known_bad.set')
    print(f"Загружено known_bad.set: {len(known_bad_set)} адресов")
except FileNotFoundError:
    # Для демонстрации создадим пустое множество
    known_bad_set = IPSet()
    known_bad_set.add(IPAddr('10.1.2.3'))
    known_bad_set.add(IPAddr('10.4.5.6'))
    print("Файл known_bad.set не найден — используем демо-множество")

# Объединение: scanners ∪ known_bad
combined_set = scanners_set | known_bad_set
combined_set.save('combined.set')
print(f"\nОбъединённое множество: {len(combined_set)} адресов → combined.set")

# Пересечение и разность — для проверки
intersection = scanners_set & known_bad_set
print(f"Пересечение (новые совпадения): {len(intersection)} адресов")

In [ ]:
# --- Шаг 5: использовать IP-множество как фильтр (аналог --sipset=scanners.set) ---
loaded_set = IPSet.load('scanners.set')

scanner_activity = []
with silkfile_open('scanner_activity.rw', WRITE) as out_f:
    for path in repo_files:
        with silkfile_open(path, READ) as in_f:
            for rec in in_f:
                if rec.sip in loaded_set:  # источник входит в множество сканеров
                    out_f.write(rec)
                    scanner_activity.append({
                        'sIP':   str(rec.sip),
                        'dIP':   str(rec.dip),
                        'dPort': rec.dport,
                        'proto': rec.protocol,
                        'bytes': rec.bytes,
                    })

df_scanners = pd.DataFrame(scanner_activity)
print(f"Потоков от сканеров: {len(df_scanners)}")
if not df_scanners.empty:
    print("\nТоп-10 целевых портов:")
    print(df_scanners['dPort'].value_counts().head(10))

---
## Пример 5. Статистика уникальных значений (`rwuniq`)

**Задача:** найти наиболее активные источники трафика по объёму переданных байт —  
аналог `rwuniq --fields=sIP --values=bytes --sort-output --reverse`.

Оригинальная команда:
```bash
rwfilter --start-date=2009/02/12 --type=out,outweb \
         --pass=stdout --fail=/dev/null | \
rwuniq --fields=sIP --values=bytes --sort-output --reverse
```

**Источник данных:** учебный набор CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# --- Шаг 1: получить исходящий трафик (типы out/outweb) ---
outbound_files = list(silk.site.repository_iter(
    start_date=TARGET_DATE,
    end_date=TARGET_DATE,
    flowtypes=['all:out', 'all:outweb']  # зависит от имён классов в silk.conf
))

# --- Шаг 2: агрегировать по sIP ---
agg_bytes    = defaultdict(int)
agg_packets  = defaultdict(int)
agg_flows    = defaultdict(int)
agg_dip_set  = defaultdict(set)

for path in outbound_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            sip = str(rec.sip)
            agg_bytes[sip]   += rec.bytes
            agg_packets[sip] += rec.packets
            agg_flows[sip]   += 1
            agg_dip_set[sip].add(str(rec.dip))

# --- Шаг 3: собрать результат и отсортировать по убыванию байт ---
df_uniq = pd.DataFrame({
    'sIP':          list(agg_bytes.keys()),
    'Bytes':        [agg_bytes[ip]         for ip in agg_bytes],
    'Packets':      [agg_packets[ip]       for ip in agg_bytes],
    'Flows':        [agg_flows[ip]         for ip in agg_bytes],
    'dIP-Distinct': [len(agg_dip_set[ip])  for ip in agg_bytes],
}).sort_values('Bytes', ascending=False).reset_index(drop=True)

print("Топ-20 источников исходящего трафика (по байтам):")
print(df_uniq.head(20).to_string(index=False))

In [ ]:
# --- Шаг 4: визуализация — кандидаты на утечку данных ---
top20 = df_uniq.head(20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top20['sIP'][::-1], top20['Bytes'][::-1] / 1e6, color='crimson', alpha=0.8)
ax.set_xlabel('Мегабайты')
ax.set_title('Топ-20 источников исходящего трафика')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('rwuniq_toptalkers.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Шаг 5: флаг потенциальной утечки — много байт + много уникальных адресатов ---
BYTES_THRESHOLD  = df_uniq['Bytes'].quantile(0.95)
DIP_THRESHOLD    = df_uniq['dIP-Distinct'].quantile(0.90)

suspects = df_uniq[
    (df_uniq['Bytes']        > BYTES_THRESHOLD) &
    (df_uniq['dIP-Distinct'] > DIP_THRESHOLD)
]
print(f"\nПодозрительных источников (высокий трафик + много адресатов): {len(suspects)}")
print(suspects.to_string(index=False))

---
## Пример 6. Конвертация pcap → SiLK через YAF и анонимизация IP

**Задача:** получить SiLK-данные из pcap-файла и при необходимости анонимизировать IP —  
аналог связки `yaf | rwipfix2silk` и утилиты `rwrandomizeip`.

Оригинальные команды:
```bash
yaf --in capture.pcap --out - | rwipfix2silk --silk-output=flows.rw
rwfileinfo flows.rw
rwrandomizeip --seed=12345 flows.rw anonymized.rw
```

**Источники:**  
- YAF (Yet Another Flowmeter): https://tools.netsa.cert.org/yaf/index.html  
- PySiLK IPAddr: https://tools.netsa.cert.org/silk/pySiLK-notes.html

In [ ]:
import subprocess

PCAP_FILE   = 'capture.pcap'
IPFIX_FILE  = 'capture.ipfix'
SILK_OUTPUT = 'flows.rw'

# --- Шаг 1: запустить yaf для генерации IPFIX из pcap ---
# yaf должен быть установлен отдельно: https://tools.netsa.cert.org/yaf/
yaf_cmd = [
    'yaf',
    '--in',  PCAP_FILE,
    '--out', IPFIX_FILE,
    '--silk',  # включить режим совместимости с SiLK
]
result = subprocess.run(yaf_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print(f"yaf: IPFIX-файл создан → {IPFIX_FILE}")
else:
    print(f"yaf завершился с ошибкой:\n{result.stderr}")

# --- Шаг 2: конвертировать IPFIX → SiLK через rwipfix2silk ---
conv_cmd = [
    'rwipfix2silk',
    f'--silk-output={SILK_OUTPUT}',
    IPFIX_FILE,
]
result2 = subprocess.run(conv_cmd, capture_output=True, text=True)
if result2.returncode == 0:
    print(f"rwipfix2silk: SiLK-файл создан → {SILK_OUTPUT}")
else:
    print(f"rwipfix2silk завершился с ошибкой:\n{result2.stderr}")

In [ ]:
# --- Шаг 3: просмотреть метаданные файла (аналог rwfileinfo) ---
with silkfile_open(SILK_OUTPUT, READ) as f:
    print(f"Формат:       {f.format}")
    print(f"Версия:       {f.version}")
    print(f"Тип данных:   {f.type}")
    print(f"Сенсор:       {f.sensor}")

# Подсчёт записей и краткая сводка
rows = []
with silkfile_open(SILK_OUTPUT, READ) as f:
    for rec in f:
        rows.append({'sIP': str(rec.sip), 'dIP': str(rec.dip),
                     'proto': rec.protocol, 'bytes': rec.bytes})

df_flows = pd.DataFrame(rows)
print(f"\nЗаписей в файле: {len(df_flows)}")
print(df_flows.head(10).to_string(index=False))

In [ ]:
import random
import hashlib

# --- Шаг 4: анонимизация IP-адресов (аналог rwrandomizeip --seed=12345) ---
# PySiLK не предоставляет rwrandomizeip напрямую;
# реализуем детерминированное псевдослучайное переименование через хэш

ANON_SEED   = 12345
ANON_OUTPUT = 'anonymized.rw'

def anonymize_ip(ip_addr, seed=ANON_SEED):
    """Детерминированная анонимизация: заменяет IP псевдослучайным адресом.
    Тот же IP всегда получает один и тот же псевдоним (для сохранения связности).
    """
    h = hashlib.md5(f"{seed}:{ip_addr}".encode()).digest()
    # Используем только первые 4 байта для IPv4
    return IPAddr(f"{h[0]}.{h[1]}.{h[2]}.{h[3]}")

with silkfile_open(ANON_OUTPUT, WRITE) as out_f:
    with silkfile_open(SILK_OUTPUT, READ) as in_f:
        for rec in in_f:
            rec.sip = anonymize_ip(rec.sip)
            rec.dip = anonymize_ip(rec.dip)
            out_f.write(rec)

print(f"Анонимизированный файл сохранён: {ANON_OUTPUT}")

# Проверка — убеждаемся, что адреса изменились
orig_sample  = []
anon_sample  = []

with silkfile_open(SILK_OUTPUT, READ) as f:
    for i, rec in enumerate(f):
        orig_sample.append(str(rec.sip))
        if i >= 4: break

with silkfile_open(ANON_OUTPUT, READ) as f:
    for i, rec in enumerate(f):
        anon_sample.append(str(rec.sip))
        if i >= 4: break

print("\nОригинальные sIP → Анонимизированные sIP:")
for orig, anon in zip(orig_sample, anon_sample):
    print(f"  {orig:20s} → {anon}")

---
## Ссылки на источники

| Ресурс | URL |
|--------|-----|
| Инструментарий SiLK (Carnegie Mellon CERT) | https://tools.netsa.cert.org/silk/ |
| Документация PySiLK | https://tools.netsa.cert.org/silk/pySiLK.html |
| Репозиторий учебных данных SiLK (CDX) | https://tools.netsa.cert.org/silk/repository.html |
| YAF (Yet Another Flowmeter) | https://tools.netsa.cert.org/yaf/index.html |
| Примеры кода к книге | https://github.com/mpcollins/nsda_examples |
| Страница книги (errata + доп. материалы) | http://bit.ly/nstda2e |